# Gradient checking with finite differences for a small neural network

We consider a binary classification problem based on two numerical features. Each point represents one observation, and the goal is to predict whether it belongs to class 0 or class 1. The two classes are arranged in a curved, nonlinearly separable pattern, so a simple linear model is not enough. In this notebook we train a **small neural network for binary classification** and use **finite differences** to check whether the implemented gradient is correct.

We work with:
- a **given 2D dataset** stored in `two_moons_nn_data.csv`,
- a neural network with **2 inputs**, **3 hidden tanh neurons**, and **1 sigmoid output**,
- the **binary cross-entropy** loss,
- an **exact gradient** obtained with the chain rule,
- a **numerical gradient** computed with central differences.


## Learning goals

By the end of the exercise you should be able to:
1. implement the forward pass of a small neural network for classification,
2. write the binary cross-entropy loss,
3. compute the gradient with respect to all parameters,
4. approximate the gradient numerically with finite differences,
5. compare both gradients using a relative error,
6. explain why gradient checking is useful in machine learning.


## 1. Load and visualize the dataset

The file `two_moons_nn_data.csv` contains three columns:
- `x1`
- `x2`
- `label`

The target `label` is binary, so this is a **classification** problem.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/two_moons_nn_data.csv')

# TODO: load the CSV file and extract:
# X = df[["x1", "x2"]].to_numpy(dtype=float)
# y = df["label"].to_numpy(dtype=float).reshape(-1, 1)

# print(df.head())
# print("X shape:", X.shape)
# print("y shape:", y.shape)

# TODO: make a scatter plot with one color per class


## 2. Preprocess the inputs

To make optimization more stable, we standardize the two input features:

$$
X_{\text{scaled}} = \frac{X - \mu}{\sigma}.
$$

We will train the network on the standardized inputs, but still visualize the data in the original coordinates.


In [ ]:
# TODO: standardize the two input features

# X_mean = ...
# X_std = ...
# X_scaled = ...

# print("Mean after scaling:", X_scaled.mean(axis=0))
# print("Std after scaling:", X_scaled.std(axis=0))


## 3. Model and parameter vector

We use the following neural network:

$$
A_1 = XW_1 + b_1,\qquad H = \tanh(A_1),
$$
$$
A_2 = HW_2 + b_2,\qquad \hat Y = \sigma(A_2),
$$

where:
- $W_1 \in \mathbb{R}^{2\times 3}$,
- $b_1 \in \mathbb{R}^{1\times 3}$,
- $W_2 \in \mathbb{R}^{3\times 1}$,
- $b_2 \in \mathbb{R}^{1\times 1}$.

To run gradient checking, we pack all parameters into a **single vector** $\theta \in \mathbb{R}^{13}$.


In [ ]:
INPUT_DIM = 2
HIDDEN_DIM = 3
OUTPUT_DIM = 1

def sigmoid(z):
    # TODO: implement the sigmoid function
    pass

def pack_parameters(W1, b1, W2, b2):
    # TODO: flatten and concatenate all parameters
    pass

def unpack_parameters(theta, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM):
    # TODO: reconstruct W1, b1, W2, b2 from the vector theta
    pass

rng = np.random.default_rng(7)
theta0 = 0.4 * rng.standard_normal(INPUT_DIM * HIDDEN_DIM + HIDDEN_DIM + HIDDEN_DIM * OUTPUT_DIM + OUTPUT_DIM)
print("Number of parameters:", len(theta0))


## 4. Forward pass and loss

For a batch of inputs $X$, implement:
1. the forward pass,
2. the binary cross-entropy loss

$$
\mathcal{L}(\theta)=
-\frac{1}{N}
\sum_{i=1}^{N}
\left[
y_i \log(\hat y_i) + (1-y_i)\log(1-\hat y_i)
\right].$$

Use a small constant `eps` inside the logarithms for numerical stability.


In [ ]:
def forward_pass(theta, X_batch):
    # TODO:
    # 1. unpack the parameters
    # 2. compute A1, H, A2, Y_hat
    # 3. return them in a dictionary
    pass

def binary_cross_entropy(theta, X_batch, y_batch, eps=1e-12):
    # TODO: implement the binary cross-entropy loss
    pass

# print("Initial loss:", binary_cross_entropy(theta0, X_scaled, y))


## 5. Exact gradient

Using the chain rule, the output-layer derivative simplifies to

$$
\frac{\partial \mathcal{L}}{\partial A_2}=
\frac{\hat Y - Y}{N}.
$$

From there, compute the gradients of all parameters:
- $dW_2$, $db_2$,
- $dW_1$, $db_1$.

Return the full gradient as a vector with the same order used in `pack_parameters`.


In [ ]:
def exact_gradient(theta, X_batch, y_batch):
    # TODO:
    # 1. compute the forward values
    # 2. implement dA2 = (Y_hat - y_batch) / N
    # 3. compute dW2, db2
    # 4. propagate to the hidden layer
    # 5. compute dW1, db1
    # 6. pack all gradients into one vector
    pass

# g0 = exact_gradient(theta0, X_scaled, y)
# print("Gradient shape:", g0.shape)
# print("First five components:", g0[:5])


## 6. Numerical gradient with central differences

For each component $\theta_i$, approximate

$$
\frac{\partial \mathcal{L}}{\partial \theta_i}
\approx
\frac{\mathcal{L}(\theta + h e_i) - \mathcal{L}(\theta - h e_i)}{2h}.
$$

This method is slow, but it is very useful for **checking** an implementation.


In [ ]:
def numerical_gradient_central(theta, X_batch, y_batch, h=1e-5):
    # TODO: implement the central-difference approximation
    pass


## 7. Gradient checking on a small batch

In practice, gradient checking is usually done on a **small batch** because it requires two loss evaluations **per parameter**.

Use the first 12 samples for the check and compare:
- the exact gradient,
- the numerical gradient,
- their absolute differences,
- the relative error

$$
\text{relative error}=
\frac{\lVert g_{\text{exact}} - g_{\text{num}} \rVert}
{\max(10^{-12}, \lVert g_{\text{exact}} \rVert + \lVert g_{\text{num}} \rVert)}.
$$


In [ ]:
X_check = X_scaled[:12]
y_check = y[:12]

# TODO:
# g_exact = ...
# g_num = ...

# comparison = pd.DataFrame({
#     "parameter_index": np.arange(len(theta0)),
#     "exact_gradient": g_exact,
#     "numerical_gradient": g_num,
#     "absolute_difference": np.abs(g_exact - g_num),
# })

# relative_error = ...

# print(comparison.head(10))
# print("\nMaximum absolute difference:", comparison["absolute_difference"].max())
# print("Relative error:", relative_error)

## 8. Train the network

Once the gradient has been checked, we can use it for optimization.

Implement full-batch gradient descent:
1. compute the exact gradient,
2. update the parameter vector,
3. store the loss history.

Use:
- `lr = 0.2`
- `n_epochs = 5000`


In [ ]:
def gradient_descent_step(theta, grad, lr):
    # TODO: update the parameter vector
    pass

def train_network(theta_init, X_train, y_train, n_epochs, lr, record_every=50):
    # TODO: implement full-batch gradient descent
    pass

# theta_fit, history = train_network(theta0, X_scaled, y, n_epochs=5000, lr=0.2)

# y_prob = forward_pass(theta_fit, X_scaled)["Y_hat"]
# y_pred = (y_prob >= 0.5).astype(int)
# accuracy = np.mean(y_pred == y)

# print("Final loss:", binary_cross_entropy(theta_fit, X_scaled, y))
# print("Training accuracy:", accuracy)


## 9. Visualize the classifier

Plot:
1. the loss history,
2. the data points,
3. the decision boundary of the trained network.

To plot the boundary in the original coordinates, remember to standardize the grid before feeding it to the network.


In [ ]:
# TODO: plot the training loss

def plot_decision_boundary(theta, X_original, y_true, X_mean, X_std, grid_points=250):
    # TODO:
    # 1. create a grid in the original coordinates
    # 2. standardize the grid
    # 3. compute the predicted probabilities
    # 4. plot the decision boundary together with the data
    pass

# plot_decision_boundary(theta_fit, X, y, X_mean, X_std)


## 10. Effect of the step size

The finite-difference approximation depends on the step size $h$.

- If $h$ is too **large**, the approximation is inaccurate because of truncation error.
- If $h$ is too **small**, rounding errors can become visible.

Test several values of $h$ and study how the relative error changes.

In [ ]:
step_sizes = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]

# TODO:
# rows = []
# for h in step_sizes:
#     g_num_h = ...
#     rel_err_h = ...
#     rows.append({
#         "h": h,
#         "relative_error": rel_err_h,
#         "max_absolute_difference": ...
#     })

# step_size_results = pd.DataFrame(rows)
# print(step_size_results)

# plt.figure(figsize=(6, 4))
# plt.loglog(step_size_results["h"], step_size_results["relative_error"], marker="o")
# plt.xlabel("Step size h")
# plt.ylabel("Relative error")
# plt.title("Effect of the finite-difference step size")
# plt.grid(True)
# plt.show()

What do you observe? In many experiments, the error decreases at first as $h$ becomes smaller, but after a point it can increase again because of floating-point errors.

## 11. A buggy gradient

Finite differences are especially useful when the analytical gradient implementation contains a mistake.

In the next cell, define a **buggy gradient** by introducing a small error in the backpropagation formula. Then compare it with the numerical gradient on the same batch.

In [ ]:
def buggy_gradient(theta, X_batch, y_batch):
    # TODO: copy the exact gradient and introduce one deliberate bug
    # For example, modify the derivative through tanh.
    pass

# TODO:
# g_bug = buggy_gradient(theta0, X_check, y_check)

# buggy_comparison = pd.DataFrame({
#     "parameter_index": np.arange(len(theta0)),
#     "buggy_gradient": g_bug,
#     "numerical_gradient": g_num,
#     "absolute_difference": np.abs(g_bug - g_num),
# })

# buggy_relative_error = ...

# print(buggy_comparison.head(10))
# print("\nMaximum absolute difference:", buggy_comparison["absolute_difference"].max())
# print("Relative error with buggy gradient:", buggy_relative_error)

## 10. Discussion

Answer briefly:

1. Why is gradient checking usually performed on a small batch rather than on the full dataset?
2. What does a very small relative error tell us?
3. Why is finite-difference gradient checking useful even when we already have an exact gradient formula?
